# [Semantic Kernel with Function Calling on native functions (plug-ins)](https://learn.microsoft.com/en-us/semantic-kernel/get-started/quick-start-guide?pivots=programming-language-python#writing-your-first-console-app)

In just a few steps, we can build our first AI agent with Semantic Kernel in either Python, .NET, or Java. This guide will show how to...

- Install the necessary packages
- Ceate a back-and-forth conversation with an AI
- Give an AI agent the ability to run your code
- Watch the AI create plans on the fly

# Constants and Libraries

In [ ]:
import os
from dotenv import load_dotenv # requires python-dotenv

load_dotenv("./../config/credentials_my.env")
print(f"os.environ['AZURE_OPENAI_ENDPOINT']: {os.environ['AZURE_OPENAI_ENDPOINT']}")

# Create an Azure chat completion object e.g. the `assistant` from SK library

In [ ]:
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion
chat_completion=AzureChatCompletion(service_id="default")
chat_completion

# Create a user message and add it to a blank history

In [ ]:
# Create a blank history of the conversation
from semantic_kernel.contents.chat_history import ChatHistory
history = ChatHistory() # initially blank

# Add user input to the history
history.add_user_message("Tell me what is Azure in less than 10 words.")

history

# Define settings for the chat
## Important: note that `function_choice_behavior=None`

In [ ]:
# Create default settings for the chat conversation

from semantic_kernel.connectors.ai.open_ai.prompt_execution_settings.azure_chat_prompt_execution_settings import (
    AzureChatPromptExecutionSettings,
)

execution_settings = AzureChatPromptExecutionSettings()
execution_settings

# Initialize the kernel

In [ ]:
from semantic_kernel import Kernel
kernel = Kernel()

kernel

# Put all together
- Chat Completion
- History (including the first user message)
- Settings
- Kernel

The `result` we get is a list of semantic_kernel.contents.chat_message_content.**ChatMessageContent** whose `content` field contains the message text.

In [ ]:
result = await chat_completion.get_chat_message_contents(
    chat_history=history,
    settings = execution_settings,
    kernel=kernel
)

#  Print the results
print(result)

print(f"\nAssistant's response: {result[0].content}")

# What does the kernel contain? No services, no plugins so far

In [ ]:
kernel

In [ ]:
# no added history too
history

# Add the Chat Completion as a service to the kernel
The Chat Completion service has id = 'default' as defined above

In [ ]:
kernel.add_service(chat_completion)

kernel 

# Add a ***Native*** Plugin to the Kernel
now the kernel has both
- a service, that we can get with kernel.get_service()
- a plugin, that we can retrieve with kernel.get_plugin("Lights")

In [ ]:
# First, we define the plugin through its class...

from typing import Annotated
from semantic_kernel.functions import kernel_function

class LightsPlugin:
    lights = [
        {"id": 0, "name": "Table Lamp", "is_on": False},
        {"id": 1, "name": "Porch light", "is_on": False},
        {"id": 2, "name": "Chandelier", "is_on": True},
    ]

    @kernel_function(
        name="get_lights", # <<<=== DIFFERENT FROM THE FUNCTION NAME <get_state>, which will be ignored
        description="Gets a list of lights and their current state",
    )
    def get_state(
        self,
    ) -> Annotated[str, "the output is a string"]:
        """Gets a list of lights and their current state."""
        return self.lights

    @kernel_function(
        name="change_state",
        description="Changes the state of the light",
    )
    def change_state(
        self,
        id: int,
        is_on: bool,
    ) -> Annotated[str, "the output is a string"]:
        """Changes the state of the light."""
        for light in self.lights:
            if light["id"] == id:
                light["is_on"] = is_on
                return light
        return None

In [ ]:
# ...then, we add the plugin to the kernel, using a new plugin name

kernel.add_plugin(
    LightsPlugin(),
    plugin_name="Lights",
)

kernel

# Create a new User message that can leverage the added plug-in

In [ ]:
# Create a blank history of the conversation
history = ChatHistory() # initially blank

# Add user input to the history
history.add_user_message("Toggle the status of my second light.")

history

# Enable planning with Function Calling set as Auto()
## Important: note that `function_choice_behavior=FunctionChoiceBehavior`

In [ ]:
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior

execution_settings = AzureChatPromptExecutionSettings()
execution_settings.function_choice_behavior= FunctionChoiceBehavior.Auto() # Auto(), Required() or NoneInvoke()
execution_settings

# Invoke the assistant using the new message and the new settings

In [ ]:
result = await chat_completion.get_chat_message_contents(
    chat_history=history,
    settings=execution_settings,
    kernel=kernel)

#  Print the results
print(result)

print(f"\nAssistant's response: {result[0].content}")

In [ ]:
history

# Additional tests. You may run the next cell multiple times to toggle the first light.

In [ ]:
# Create a blank history of the conversation
from semantic_kernel.contents.chat_history import ChatHistory
history = ChatHistory() # initially blank
history.add_user_message("Toggle the first light and give me the status of all my lights.")

result = await chat_completion.get_chat_message_contents(
    chat_history=history,
    settings=execution_settings,
    kernel=kernel)

#  Print the results
print(f"\nAssistant's response: {result[0].content}")

In [ ]:
for cmc in history.messages: # ChatMessageContent
    if not cmc.inner_content is None:
        for choice in cmc.inner_content.choices:
            for tc in choice.message.tool_calls:
                print (f"Call {tc.function.name}({tc.function.arguments})")

In [ ]:
history